# Sprint 20 — Generate 4 Datasets for V9 Pipeline

**Mục tiêu:** Sinh đủ 4 datasets với seed cố định cho pipeline V9.

## 4-Dataset Strategy

| Dataset | Nội dung | Dùng cho |
|---------|----------|----------|
| `train_welding_robot.csv` | Normal only, reduced noise, deterministic | Training model |
| `val_train_welding_robot.csv` | Normal only, full noise (7 ngày) | Validate loss trong training loop |
| `threshold_calibration_welding_robot.csv` | Normal + 5 anomaly types, labelled | Chọn threshold thủ công |
| `test_welding_robot.csv` | Normal + 5 anomaly types, labelled | Evaluation cuối |

**Seed cố định:** `seed=42` cho mọi BatchGenerator.
**Output dir:** `data/generated_sprint20/`

In [1]:
# %% Imports
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
# Walk up to find project root
for parent in [project_root] + list(project_root.parents):
    if (parent / "backend").is_dir():
        project_root = parent
        break
# Add backend/ to sys.path so we can import data_simulator directly
backend_path = str(project_root / "backend")
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)
print(f"Backend path: {backend_path}")

from data_simulator.batch_generator import BatchGenerator
from data_simulator.writer import CSVWriter
from datetime import datetime

print("All imports OK")

Backend path: /home/trietlm/TED-Hackathon-Agent/backend
All imports OK


## Dataset 1: Training Data (Normal Only, Reduced Noise)

Dùng `BatchGenerator(training_mode=True)`:
- Normal only — không có anomaly nào
- Reduced noise + deterministic (cùng robot_id, program_id → cùng envelope)
- 60 ngày, 4 robots
- Dùng để training model

In [2]:
# %% Generate training data (convergent mode, reduced noise)
import os

output_dir = str(project_root / "data" / "generated_sprint20")
os.makedirs(output_dir, exist_ok=True)
print(f"Output dir: {output_dir}")

train_gen = BatchGenerator(seed=42, training_mode=True)
writer = CSVWriter(output_dir=output_dir)

print("Generating training data (60 days, 4 robots, convergent, reduced-noise)...")
train_path, train_count = writer.write_train(
    train_gen,
    num_days=60,
    robots=[0, 1, 2, 3],
    start_date=datetime(2026, 1, 1),
)
print(f"Training data: {train_count:,} rows -> {train_path}")

Output dir: /home/trietlm/TED-Hackathon-Agent/data/generated_sprint20
Generating training data (60 days, 4 robots, convergent, reduced-noise)...
Training data: 32,769,024 rows -> /home/trietlm/TED-Hackathon-Agent/data/generated_sprint20/train_welding_robot.csv


## Dataset 2: Validation Training Data (Normal Only, Full Noise)

Dùng `BatchGenerator(seed=1023)` — **KHÔNG** `training_mode`:
- Normal only — không có anomaly
- **Full noise** — realistic sensor noise để validation loss phản ánh đúng điều kiện thực tế
- 7 ngày, 4 robots
- Dùng để validate loss trong training loop

In [3]:
# %% Generate validation training data (normal only, full noise)
val_train_gen = BatchGenerator(seed=1023)  # NOT training_mode => full noise
val_train_path, val_train_count = writer.write_train(
    val_train_gen,
    num_days=15,
    robots=[0, 1, 2, 3],
    start_date=datetime(2026, 3, 1),
    output_filename="val_train_welding_robot.csv",
)
print(f"Validation training data: {val_train_count:,} rows -> {val_train_path}")

Validation training data: 8,139,264 rows -> /home/trietlm/TED-Hackathon-Agent/data/generated_sprint20/val_train_welding_robot.csv


## Dataset 3: Threshold Calibration Data (Normal + Anomaly, Labelled)

Dùng `writer.write_threshold_calibration()`:
- 50% normal, 50% anomalous (5 types: spike, thermal_drift, flatline, cable_wear, arc_blow)
- Có labels: `threshold_calibration_labels.csv`
- Dùng để chọn threshold thủ công: plot distribution per sensor × per anomaly type
- Chọn threshold sao cho normal scores nằm dưới threshold và anomaly scores nằm trên

In [4]:
# %% Generate threshold calibration data (normal + anomaly, labelled)
calib_gen = BatchGenerator(seed=0)
calib_csv_path, calib_label_path, calib_count = writer.write_threshold_calibration(
    calib_gen,
    num_cycles=8000,
    anomaly_ratio=0.3,
    robots=[0, 1, 2, 3],
    start_date=datetime(2026, 4, 1),
    seed=0,
)
print(f"Threshold calibration data: {calib_count:,} rows -> {calib_csv_path}")
print(f"Calibration labels: -> {calib_label_path}")

Threshold calibration data: 4,096,000 rows -> /home/trietlm/TED-Hackathon-Agent/data/generated_sprint20/threshold_calibration_welding_robot.csv
Calibration labels: -> /home/trietlm/TED-Hackathon-Agent/data/generated_sprint20/threshold_calibration_labels.csv


## Dataset 4: Test Data (Normal + Anomaly, Labelled)

Dùng `writer.write_test()`:
- 50% normal, 50% anomalous (5 types)
- Có labels: `test_labels.csv`
- Dùng cho evaluation cuối: tính AUROC, precision, recall, F1

In [5]:
# %% Generate test data (normal + anomaly, labelled)
test_gen = BatchGenerator(seed=123)
test_csv_path, test_label_path, test_count = writer.write_test(
    test_gen,
    num_cycles=8000,
    anomaly_ratio=0.5,
    robots=[0, 1, 2, 3],
    seed=123,
    start_date=datetime(2026, 5, 1),
)
print(f"Test data: {test_count:,} rows -> {test_csv_path}")
print(f"Test labels: -> {test_label_path}")

Test data: 4,096,000 rows -> /home/trietlm/TED-Hackathon-Agent/data/generated_sprint20/test_welding_robot.csv
Test labels: -> /home/trietlm/TED-Hackathon-Agent/data/generated_sprint20/test_labels.csv


## Verification: Label Distribution & Step Counts

Kiểm tra:
1. Mỗi dataset có đúng số cycles và steps
2. Anomaly types phân bố đều trong threshold_calibration và test
3. Mọi cycle đều có 512 steps
4. Chỉ 5 anomaly types + 'normal' xuất hiện (không có archived types)

In [6]:
# %% Verify label distribution and step counts
import pandas as pd
import numpy as np

# Load labels for calibration and test sets
calib_labels = pd.read_csv(calib_label_path)
test_labels = pd.read_csv(test_label_path)

print("=== THRESHOLD CALIBRATION LABEL DISTRIBUTION ===")
print(calib_labels["anomaly_type"].value_counts())
print(f"Normal: {(calib_labels['is_anomaly'] == 0).sum()}")
print(f"Anomaly: {(calib_labels['is_anomaly'] == 1).sum()}")
print()

print("=== TEST LABEL DISTRIBUTION ===")
print(test_labels["anomaly_type"].value_counts())
print(f"Normal: {(test_labels['is_anomaly'] == 0).sum()}")
print(f"Anomaly: {(test_labels['is_anomaly'] == 1).sum()}")
print()

# Verify step counts
for name, path in [
    ("Train", train_path),
    ("Val Train", val_train_path),
    ("Threshold Calib", calib_csv_path),
    ("Test", test_csv_path),
]:
    df = pd.read_csv(path)
    steps = df.groupby("cycle_id").size()
    min_s, max_s, uniq = steps.min(), steps.max(), steps.nunique()
    status = "OK" if (min_s == 512 and max_s == 512 and uniq == 1) else "FAIL"
    print(f"{name:20s}: {len(df):>8,} rows, {steps.nunique()} unique step counts, "
          f"min={min_s}, max={max_s} [{status}]")

# Verify anomaly types (only 5 + normal)
test_anom_types = set(test_labels[test_labels["is_anomaly"] == 1]["anomaly_type"].unique())
calib_anom_types = set(calib_labels[calib_labels["is_anomaly"] == 1]["anomaly_type"].unique())
print(f"\nTest anomaly types: {sorted(test_anom_types)} ({len(test_anom_types)})")
print(f"Calib anomaly types: {sorted(calib_anom_types)} ({len(calib_anom_types)})")

archived = {"phase_shift", "wire_stutter", "gas_blockage", "sticky_tc", "overshoot_degradation"}
for name, types in [("Test", test_anom_types), ("Calib", calib_anom_types)]:
    found = types & archived
    if found:
        print(f"WARNING [{name}]: Archived types found: {found}")
    else:
        print(f"OK [{name}]: No archived types")

print("\n=== ALL VERIFICATIONS PASSED ===")

=== THRESHOLD CALIBRATION LABEL DISTRIBUTION ===
anomaly_type
normal           5590
flatline          497
arc_blow          489
spike             485
thermal_drift     482
cable_wear        457
Name: count, dtype: int64
Normal: 5590
Anomaly: 2410

=== TEST LABEL DISTRIBUTION ===
anomaly_type
normal           4022
arc_blow          835
spike             829
flatline          792
thermal_drift     778
cable_wear        744
Name: count, dtype: int64
Normal: 4022
Anomaly: 3978

Train               : 32,769,024 rows, 1 unique step counts, min=512, max=512 [OK]
Val Train           : 8,139,264 rows, 1 unique step counts, min=512, max=512 [OK]
Threshold Calib     : 4,096,000 rows, 1 unique step counts, min=512, max=512 [OK]
Test                : 4,096,000 rows, 1 unique step counts, min=512, max=512 [OK]

Test anomaly types: ['arc_blow', 'cable_wear', 'flatline', 'spike', 'thermal_drift'] (5)
Calib anomaly types: ['arc_blow', 'cable_wear', 'flatline', 'spike', 'thermal_drift'] (5)
OK [Test]: N